> **Note:** originally developed and run in Google Colab. Google-Drive mounting cells were removed, Drive paths were rewritten as relative paths and outputs were stripped (figures of the original run are in `assets/`). The paths/cleanup were **not re-executed** after this change, so expect to adjust paths.


# Fine-tuning UNet para segmentación semántica (Colab + GPU)

Este cuaderno entrena un modelo **UNet** (con *encoder* EfficientNet-B0 preentrenado en ImageNet) para **segmentación binaria** de embriones.
Está pensado para Google Colab con GPU y asume la siguiente estructura en tu Google Drive:

```
/MyDrive/Embrio_RRNN_OK/
  ├── Images/  (p.ej. emb_001.JPG, emb_002.JPG, ...)
  └── Masks/   (p.ej. emb_001.png, emb_002.png, ...)
```
Las máscaras deben ser **binarias** (fondo=0, objeto=255 ó 0/1). Si tienen 0/255, el cuaderno normaliza a 0/1 automáticamente.


In [ ]:

#@title (1) Comprobación de GPU y versiones
import torch, platform, sys
print("Python :", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA   :", torch.version.cuda if torch.cuda.is_available() else None)
print("GPU    :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No-GPU")


In [ ]:
#@title (2) Instalar dependencias (ejecutar una vez por sesión)
!pip -q install -U segmentation-models-pytorch==0.3.3 albumentations==1.4.18 opencv-python==4.10.0.84 timm==0.9.2 torchsummary==1.5.1 albucore==0.0.17


In [ ]:

#@title (3) Importar librerías
import os
from pathlib import Path
import random
import cv2
import numpy as np
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

import segmentation_models_pytorch as smp
from torchsummary import summary

# Reproducibilidad
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


In [ ]:

#@title (4) Montar Google Drive y definir rutas

ROOT_DIR = Path('.')
IMAGES_DIR = ROOT_DIR / 'Images'
MASKS_DIR  = ROOT_DIR / 'Masks'
OUT_DIR = ROOT_DIR / "FineTuning_Outputs"

assert IMAGES_DIR.exists(), f"No existe {IMAGES_DIR}"
assert MASKS_DIR.exists(),  f"No existe {MASKS_DIR}"

print("Images:", IMAGES_DIR)
print("Masks :", MASKS_DIR)


In [ ]:

#@title (5) Hiperparámetros y configuración
IMG_SIZE = 1024  #@param {type:"integer"}
BATCH_SIZE = 2  #@param {type:"integer"}
EPOCHS = 50     #@param {type:"integer"}
LR = 3e-4       #@param {type:"number"}
WEIGHT_DECAY = 1e-5  #@param {type:"number"}

ENCODER = "timm-efficientnet-b0"  #@param ["timm-efficientnet-b0", "resnet34", "resnet50", "timm-mobilenetv3_large_100"]
ENCODER_WEIGHTS = "imagenet"      #@param ["imagenet", "None"]
CLASSES = 1  # binaria
ACTIVATION = None  # logits

OUT_DIR = ROOT_DIR / "FineTuning_Outputs"
CKPT_DIR = OUT_DIR / "checkpoints"
PRED_DIR = OUT_DIR / "preds"
for d in [OUT_DIR, CKPT_DIR, PRED_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Salida:", OUT_DIR)


In [ ]:

#@title (6) Dataset y Dataloaders
def list_pairs(images_dir: Path, masks_dir: Path):
    # Emparejar por nombre (emb_001.JPG -> emb_001.png)
    img_paths = []
    msk_paths = []
    for img_path in sorted(images_dir.glob("*")):
        stem = img_path.stem  # emb_001
        # Buscar la máscara con el mismo stem pero extensión .png
        msk_path = masks_dir / f"{stem}.png"
        if msk_path.exists():
            img_paths.append(img_path)
            msk_paths.append(msk_path)
    return img_paths, msk_paths

class EmbryoSegDataset(Dataset):
    def __init__(self, imgs, msks, transform=None):
        self.imgs = imgs
        self.msks = msks
        self.transform = transform

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        img_path = str(self.imgs[idx])
        msk_path = str(self.msks[idx])

        img = cv2.imread(img_path, cv2.IMREAD_COLOR)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        msk = cv2.imread(msk_path, cv2.IMREAD_GRAYSCALE)
        # Normalizar máscara a {0,1}
        msk = (msk > 127).astype(np.float32)

        if self.transform:
            augmented = self.transform(image=img, mask=msk)
            img = augmented["image"]
            msk = augmented["mask"].unsqueeze(0)  # [1, H, W]
        else:
            img = torch.tensor(img).permute(2,0,1).float()/255.0
            msk = torch.tensor(msk).unsqueeze(0).float()

        return img, msk

train_tfms = A.Compose([
    A.LongestMaxSize(max_size=IMG_SIZE),
    A.PadIfNeeded(IMG_SIZE, IMG_SIZE, border_mode=cv2.BORDER_CONSTANT, value=0, mask_value=0),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.RandomRotate90(p=0.2),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.15, rotate_limit=20, border_mode=cv2.BORDER_REFLECT, p=0.5),
    A.GaussianBlur(blur_limit=(3,5), p=0.2),
    A.ColorJitter(p=0.2),
    A.Normalize(),
    ToTensorV2(),
])

valid_tfms = A.Compose([
    A.LongestMaxSize(max_size=IMG_SIZE),
    A.PadIfNeeded(IMG_SIZE, IMG_SIZE, border_mode=cv2.BORDER_CONSTANT, value=0, mask_value=0),
    A.Normalize(),
    ToTensorV2(),
])

imgs, msks = list_pairs(IMAGES_DIR, MASKS_DIR)
print(f"Total pares encontrados: {len(imgs)}")

# Split reproducible 80/20
indices = list(range(len(imgs)))
random.Random(SEED).shuffle(indices)
split = int(0.8 * len(indices))
train_idx, valid_idx = indices[:split], indices[split:]

train_imgs = [imgs[i] for i in train_idx]
train_msks = [msks[i] for i in train_idx]
valid_imgs = [imgs[i] for i in valid_idx]
valid_msks = [msks[i] for i in valid_idx]

train_ds = EmbryoSegDataset(train_imgs, train_msks, transform=train_tfms)
valid_ds = EmbryoSegDataset(valid_imgs, valid_msks, transform=valid_tfms)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

len(train_ds), len(valid_ds)


In [ ]:

#@title (7) Modelo UNet (SMP) + resumen
model = smp.Unet(
    encoder_name=ENCODER,
    encoder_weights=None if ENCODER_WEIGHTS == "None" else ENCODER_WEIGHTS,
    in_channels=3,
    classes=CLASSES,
)

model.to(device)
summary(model, (3, IMG_SIZE, IMG_SIZE))


In [ ]:

#@title (8) Funciones de pérdida y métricas (Dice / IoU) sin torchmetrics
bce = nn.BCEWithLogitsLoss()

def dice_coeff(pred_logits, target, eps=1e-7):
    # pred_logits: [B,1,H,W] (logits); target: [B,1,H,W] (0/1)
    probs = torch.sigmoid(pred_logits)
    probs = (probs > 0.5).float()
    inter = (probs * target).sum(dim=(1,2,3))
    union = probs.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3))
    dice = (2*inter + eps) / (union + eps)
    return dice.mean()

def iou_score(pred_logits, target, eps=1e-7):
    probs = torch.sigmoid(pred_logits)
    probs = (probs > 0.5).float()
    inter = (probs * target).sum(dim=(1,2,3))
    union = probs.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3)) - inter
    iou = (inter + eps) / (union + eps)
    return iou.mean()

def loss_fn(pred_logits, target):
    # BCE + Dice (suavizado)
    probs = torch.sigmoid(pred_logits)
    # Dice suave (sin binarizar)
    inter = (probs * target).sum(dim=(1,2,3))
    union = probs.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3))
    dice_loss = 1 - (2*inter + 1) / (union + 1)
    return bce(pred_logits, target) + dice_loss.mean()


In [ ]:

#@title (9) Optimizador, AMP y scheduler
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=3
)
scaler = torch.cuda.amp.GradScaler(enabled=(device.type=="cuda"))


In [ ]:

#@title (10) Entrenamiento
from time import time

best_val = float('inf')
history = {"train_loss": [], "val_loss": [], "val_dice": [], "val_iou": []}

for epoch in range(1, EPOCHS+1):
    t0 = time()
    model.train()
    train_losses = []

    for imgs, msks in train_loader:
        imgs = imgs.to(device, non_blocking=True)
        msks = msks.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=(device.type=="cuda")):
            logits = model(imgs)
            loss = loss_fn(logits, msks)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_losses.append(loss.item())

    # Validación
    model.eval()
    val_losses = []
    dices, ious = [], []
    with torch.no_grad():
        for imgs, msks in valid_loader:
            imgs = imgs.to(device, non_blocking=True)
            msks = msks.to(device, non_blocking=True)
            logits = model(imgs)
            loss = loss_fn(logits, msks)
            val_losses.append(loss.item())
            dices.append(dice_coeff(logits, msks).item())
            ious.append(iou_score(logits, msks).item())

    train_loss = float(np.mean(train_losses)) if train_losses else np.nan
    val_loss = float(np.mean(val_losses)) if val_losses else np.nan
    val_dice = float(np.mean(dices)) if dices else np.nan
    val_iou  = float(np.mean(ious)) if ious else np.nan

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_dice"].append(val_dice)
    history["val_iou"].append(val_iou)

    scheduler.step(val_loss)

    # Guardar mejor checkpoint (por menor val_loss)
    if val_loss < best_val:
        best_val = val_loss
        ckpt_path = CKPT_DIR / "best_unet.pth"
        torch.save({"state_dict": model.state_dict(),
                    "encoder": ENCODER,
                    "img_size": IMG_SIZE}, ckpt_path)
        best_msg = f"  ✅ Guardado mejor modelo: {ckpt_path.name}"
    else:
        best_msg = ""

    dt = time()-t0
    print(f"[{epoch:02d}/{EPOCHS}] train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | "
          f"Dice={val_dice:.4f} | IoU={val_iou:.4f} | {dt:.1f}s{best_msg}")


In [ ]:

#@title (11) Curvas de entrenamiento
plt.figure(figsize=(10,4))
plt.plot(history["train_loss"], label="train_loss")
plt.plot(history["val_loss"], label="val_loss")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend(); plt.title("Curva de Loss"); plt.show()

plt.figure(figsize=(10,4))
plt.plot(history["val_dice"], label="val_dice")
plt.plot(history["val_iou"], label="val_iou")
plt.xlabel("Epoch"); plt.ylabel("Score"); plt.legend(); plt.title("Métricas de Validación"); plt.show()


In [ ]:
#@title (12) Inferencia, overlays y guardado de predicciones
import pandas as pd

# Cargar mejor modelo
ckpt = torch.load(CKPT_DIR / "best_unet.pth", map_location=device)
model.load_state_dict(ckpt["state_dict"])
model.eval()

# Directorio para predicciones
PRED_DIR.mkdir(parents=True, exist_ok=True)

def _fit_to_square_meta(h, w, size):
    # escala y offsets si primero escalamos por el lado mayor y luego centramos con padding
    s = size / max(h, w)
    nh, nw = int(round(h * s)), int(round(w * s))   # tamaño tras el resize manteniendo aspecto
    top  = (size - nh) // 2
    left = (size - nw) // 2
    return s, nh, nw, top, left

def predict_mask(img_np):
    # img_np: RGB uint8 [H,W,3]
    H, W = img_np.shape[:2]
    s, nh, nw, top, left = _fit_to_square_meta(H, W, IMG_SIZE)

    # 1) resize manteniendo aspecto
    img_res = cv2.resize(img_np, (nw, nh), interpolation=cv2.INTER_AREA)
    # 2) pad a cuadrado centrado
    canvas = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
    canvas[top:top+nh, left:left+nw] = img_res

    # 3) normalizar y tensor
    tfm = A.Compose([A.Normalize(), ToTensorV2()])
    tens = tfm(image=canvas)["image"].unsqueeze(0).to(device)

    # 4) predicción cuadrada
    with torch.no_grad():
        logits = model(tens)
        probs = torch.sigmoid(logits)
        pred_sq = (probs > 0.5).float().cpu().numpy()[0, 0].astype(np.uint8)  # 0/1

    # 5) quitar padding y volver al tamaño original
    pred_cropped = pred_sq[top:top+nh, left:left+nw]
    pred_orig = cv2.resize(pred_cropped, (W, H), interpolation=cv2.INTER_NEAREST)  # 0/1

    return (pred_orig * 255).astype(np.uint8)

def overlay(img, mask_bin, alpha=0.4):
    # img: RGB uint8 [H,W,3]; mask_bin: 0/255 ya alineada a [H,W]
    if mask_bin.shape[:2] != img.shape[:2]:
        mask_bin = cv2.resize(mask_bin, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST)
    color = np.zeros_like(img)
    color[:, :, 0] = mask_bin  # canal R
    return cv2.addWeighted(img, 1 - alpha, color, alpha, 0)

pred_map = []
for img_path in valid_imgs:
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    pred_mask = predict_mask(img)
    out_name = Path(img_path).stem + "_pred.png"
    cv2.imwrite(str(PRED_DIR / out_name), pred_mask)

    # Guardar overlay de muestra
    ov = overlay(img, pred_mask)
    ov_name = Path(img_path).stem + "_overlay.jpg"
    cv2.imwrite(str(PRED_DIR / ov_name), cv2.cvtColor(ov, cv2.COLOR_RGB2BGR))

    pred_map.append({"image": str(img_path), "pred_mask": str(PRED_DIR / out_name)})

pred_df = pd.DataFrame(pred_map)
csv_path = PRED_DIR / "pred_map.csv"
pred_df.to_csv(csv_path, index=False)
print("Predicciones guardadas en:", PRED_DIR)
print("Mapa pred->img:", csv_path)

In [ ]:

#@title (13) Visualizador rápido de predicciones (validación)
def show_pred_overlays(n=6):
    shown = 0
    plt.figure(figsize=(12, 10))
    for i, img_path in enumerate(valid_imgs):
        img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        pred = cv2.imread(str(PRED_DIR / (Path(img_path).stem + "_pred.png")), cv2.IMREAD_GRAYSCALE)
        if pred is None:
            continue
        ov = overlay(img, pred)
        plt.subplot(2, max(1, n//2) if n>2 else n, i+1)
        plt.imshow(ov); plt.axis("off"); plt.title(Path(img_path).stem)
        shown += 1
        if shown >= n:
            break
    plt.tight_layout(); plt.show()

show_pred_overlays(40)


In [ ]:
# ==== (A) Paths de entrada/salida ====
RESTO_DIR = ROOT_DIR / "Resto"
OUT_RESTO = OUT_DIR / "preds_resto"
OUT_RESTO.mkdir(parents=True, exist_ok=True)

assert RESTO_DIR.exists(), f"No existe la carpeta: {RESTO_DIR}"

def _fit_to_square_meta(h, w, size):
    # escala y offsets si primero escalamos por el lado mayor y luego centramos con padding
    s = size / max(h, w)
    nh, nw = int(round(h * s)), int(round(w * s))   # tamaño tras el resize manteniendo aspecto
    top  = (size - nh) // 2
    left = (size - nw) // 2
    return s, nh, nw, top, left

def predict_mask(img_np):
    # img_np: RGB uint8 [H,W,3]
    H, W = img_np.shape[:2]
    s, nh, nw, top, left = _fit_to_square_meta(H, W, IMG_SIZE)

    # 1) resize manteniendo aspecto
    img_res = cv2.resize(img_np, (nw, nh), interpolation=cv2.INTER_AREA)
    # 2) pad a cuadrado centrado
    canvas = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
    canvas[top:top+nh, left:left+nw] = img_res

    # 3) normalizar y tensor
    tfm = A.Compose([A.Normalize(), ToTensorV2()])
    tens = tfm(image=canvas)["image"].unsqueeze(0).to(device)

    # 4) predicción cuadrada
    with torch.no_grad():
        logits = model(tens)
        probs = torch.sigmoid(logits)
        pred_sq = (probs > 0.5).float().cpu().numpy()[0, 0].astype(np.uint8)  # 0/1

    # 5) quitar padding y volver al tamaño original
    pred_cropped = pred_sq[top:top+nh, left:left+nw]
    pred_orig = cv2.resize(pred_cropped, (W, H), interpolation=cv2.INTER_NEAREST)  # 0/1

    return (pred_orig * 255).astype(np.uint8)

def overlay(img, mask_bin, alpha=0.4):
    # img: RGB uint8 [H,W,3]; mask_bin: 0/255 ya alineada a [H,W]
    if mask_bin.shape[:2] != img.shape[:2]:
        mask_bin = cv2.resize(mask_bin, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST)
    color = np.zeros_like(img)
    color[:, :, 0] = mask_bin  # canal R
    return cv2.addWeighted(img, 1 - alpha, color, alpha, 0)

IMG_SIZE = 1024  #@param {type:"integer"}

# Extensiones aceptadas
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".JPG", ".PNG"}

# ==== (B) Listar imágenes ====
resto_imgs = sorted([p for p in RESTO_DIR.iterdir() if p.suffix in IMG_EXTS])
print(f"Imágenes encontradas en Resto: {len(resto_imgs)}")

# ==== (C) Función para procesar y visualizar ====
import matplotlib.pyplot as plt
import cv2
from pathlib import Path
import numpy as np

def predict_and_show_resto(n_to_show=6, save_outputs=True, alpha=0.4):
    """
    Predice y visualiza sobre la carpeta Resto.
    n_to_show: cuántas tripletas (original/máscara/overlay) mostrar.
    save_outputs: si True, guarda máscara y overlay en OUT_RESTO.
    """
    count = 0
    for img_path in resto_imgs:
        # Leer RGB
        img_bgr = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
        if img_bgr is None:
            continue
        img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        # Predicción (0/255) del mismo tamaño que la imagen original
        pred_mask = predict_mask(img)  # usa tu función con recorte de padding
        ov = overlay(img, pred_mask, alpha=alpha)

        # Guardar
        if save_outputs:
            stem = Path(img_path).stem
            cv2.imwrite(str(OUT_RESTO / f"{stem}_pred.png"), pred_mask)                 # binaria 0/255
            cv2.imwrite(str(OUT_RESTO / f"{stem}_overlay.jpg"), cv2.cvtColor(ov, cv2.COLOR_RGB2BGR))

        # Visualizar (hasta n_to_show)
        if count < n_to_show:
            plt.figure(figsize=(14, 5))
            plt.suptitle(stem, fontsize=14)

            plt.subplot(1, 3, 1)
            plt.imshow(img); plt.axis("off"); plt.title("Original")

            plt.subplot(1, 3, 2)
            plt.imshow(pred_mask, cmap="gray", vmin=0, vmax=255)
            plt.axis("off"); plt.title("Máscara predicha")

            plt.subplot(1, 3, 3)
            plt.imshow(ov); plt.axis("off"); plt.title("Overlay")

            plt.tight_layout(rect=[0, 0, 1, 0.95])
            plt.show()

        count += 1

    print(f"Procesadas {count} imágenes. Salida: {OUT_RESTO}")

# Ejecutar
predict_and_show_resto(n_to_show=48, save_outputs= True, alpha=0.4)

# **Cargar el modelo**

In [ ]:
import torch
import segmentation_models_pytorch as smp
from pathlib import Path
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Montar drive
try:
except:

# 1. Definir el dispositivo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Ruta del modelo guardado
CKPT_PATH = Path("./FineTuning_Outputs/checkpoints/best_unet.pth")

# Cargar checkpoint
ckpt = torch.load(CKPT_PATH, map_location=device)

# Variable clave que necesita la función de predicción
IMG_SIZE = ckpt.get("img_size", 1024)

# Crear modelo con la misma arquitectura
model = smp.Unet(
    encoder_name=ckpt["encoder"],
    encoder_weights=None,
    in_channels=3,
    classes=1,
)

# Cargar pesos
model.load_state_dict(ckpt["state_dict"])

# --- LA SOLUCIÓN ---
# Mover el modelo al dispositivo (GPU si está disponible)
model.to(device)
# --------------------

# Poner en modo evaluación
model.eval()

print(f"✅ Modelo cargado: {ckpt['encoder']} | Img_size: {IMG_SIZE} | Device: {device}")

In [ ]:
def _fit_to_square_meta(h, w, size):
    """Calcula cómo ajustar una imagen (h, w) a un cuadrado (size, size) con padding."""
    s = size / max(h, w)
    nh, nw = int(round(h * s)), int(round(w * s))
    top  = (size - nh) // 2
    left = (size - nw) // 2
    return s, nh, nw, top, left

def predict_mask(img_np):
    """Toma una imagen NumPy [H,W,3] y devuelve una máscara binaria [H,W]."""
    # img_np: RGB uint8 [H,W,3]
    H, W = img_np.shape[:2]

    # 1. Calcular padding y resize
    s, nh, nw, top, left = _fit_to_square_meta(H, W, IMG_SIZE)

    # 2. Resize manteniendo aspecto
    img_res = cv2.resize(img_np, (nw, nh), interpolation=cv2.INTER_AREA)

    # 3. Pad a cuadrado centrado
    canvas = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
    canvas[top:top+nh, left:left+nw] = img_res

    # 4. Normalizar y convertir a tensor
    tfm = A.Compose([A.Normalize(), ToTensorV2()])
    tens = tfm(image=canvas)["image"].unsqueeze(0).to(device) # [1, 3, H, W]

    # 5. Predicción
    with torch.no_grad():
        logits = model(tens)
        probs = torch.sigmoid(logits)
        # Binarizar y pasar a CPU/Numpy
        pred_sq = (probs > 0.5).float().cpu().numpy()[0, 0].astype(np.uint8) # [IMG_SIZE, IMG_SIZE]

    # 6. Quitar padding
    pred_cropped = pred_sq[top:top+nh, left:left+nw]

    # 7. Volver al tamaño original
    pred_orig = cv2.resize(pred_cropped, (W, H), interpolation=cv2.INTER_NEAREST) # 0/1

    return (pred_orig * 255).astype(np.uint8) # 0/255

def overlay(img, mask_bin, alpha=0.4):
    """Superpone una máscara roja sobre una imagen RGB."""
    # img: RGB uint8 [H,W,3]; mask_bin: 0/255 [H,W]
    if mask_bin.shape[:2] != img.shape[:2]:
        mask_bin = cv2.resize(mask_bin, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST)

    color = np.zeros_like(img)
    color[:, :, 0] = mask_bin  # Poner la máscara en el canal Rojo

    return cv2.addWeighted(img, 1 - alpha, color, alpha, 0)

In [ ]:
# Define tus rutas base
ROOT_DIR = Path('.')
OUT_DIR = ROOT_DIR / "FineTuning_Outputs"

# 1. Define la carpeta con TUS NUEVAS imágenes
NUEVAS_IMAGENES_DIR = ROOT_DIR / "Resto" # <--- CAMBIA ESTO si tus imágenes están en otra parte

# 2. Define dónde guardar los resultados
SALIDA_PREDICCIONES_DIR = OUT_DIR / "preds_nuevas"
SALIDA_PREDICCIONES_DIR.mkdir(parents=True, exist_ok=True)

# 3. Listar imágenes
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".JPG", ".PNG"}
lista_imagenes = sorted([p for p in NUEVAS_IMAGENES_DIR.iterdir() if p.suffix in IMG_EXTS])

print(f"Ruta de entrada: {NUEVAS_IMAGENES_DIR}")
print(f"Ruta de salida: {SALIDA_PREDICCIONES_DIR}")
print(f"Imágenes nuevas encontradas: {len(lista_imagenes)}")

In [ ]:
# --- Configuración de Inferencia ---
N_A_MOSTRAR = 10
ALPHA_OVERLAY = 0.4

# --- TU ESCALA ---
# Escala lineal: 317.4291 px = 1 cm
ESCALA_LINEAL_PX_POR_CM = 317.4291
# Escala de área: (px/cm)^2 = px^2 / cm^2
ESCALA_AREA_PX2_POR_CM2 = ESCALA_LINEAL_PX_POR_CM ** 2
print(f"Using scale: 1 cm² = {ESCALA_AREA_PX2_POR_CM2:.2f} px²")
# ---

# Lista para guardar los resultados (para el CSV)
resultados = []

count = 0
for img_path in lista_imagenes:

    # 1. Leer la imagen original en RGB
    img_bgr = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
    if img_bgr is None:
        print(f"Warning: No se pudo leer {img_path}")
        continue
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # 2. Generar la predicción (0/255)
    pred_mask = predict_mask(img_rgb)

    # 3. Calcular el área en píxeles
    area_en_pixeles = cv2.countNonZero(pred_mask)

    # 4. Convertir a área real (cm²) <-- MODIFICADO
    area_en_cm2 = area_en_pixeles / ESCALA_AREA_PX2_POR_CM2

    # 5. Crear el overlay y guardar
    img_overlay = overlay(img_rgb, pred_mask, alpha=ALPHA_OVERLAY)
    stem = img_path.stem
    cv2.imwrite(str(SALIDA_PREDICCIONES_DIR / f"{stem}_pred.png"), pred_mask)
    # ESTA ES LA LÍNEA CORRECTA:
    cv2.imwrite(str(SALIDA_PREDICCIONES_DIR / f"{stem}_overlay.jpg"), cv2.cvtColor(img_overlay, cv2.COLOR_RGB2BGR))

    # 6. Guardar datos en la lista
    resultados.append({
        "imagen": img_path.name,
        "area_pixeles": area_en_pixeles,
        "area_cm2": area_en_cm2
    })

    # 7. Visualizar
    if count < N_A_MOSTRAR:
        plt.figure(figsize=(15, 6))

        plt.subplot(1, 3, 1)
        plt.imshow(img_rgb); plt.title("Original"); plt.axis("off")

        plt.subplot(1, 3, 2)
        plt.imshow(pred_mask, cmap="gray"); plt.title("Predicted Mask"); plt.axis("off")

        plt.subplot(1, 3, 3)
        plt.imshow(img_overlay); plt.title("Overlay"); plt.axis("off")

        # Título con ambas áreas (usamos 4 decimales para cm²) <-- MODIFICADO
        titulo_plot = f"Image:{stem}\nArea: {area_en_pixeles} px²  ({area_en_cm2:.4f} cm²)"
        plt.suptitle(titulo_plot, fontsize=16)

        plt.tight_layout(rect=[0, 0, 1, 0.93])
        plt.show()

    # Imprimir en consola <-- MODIFICADO
    print(f"Procesada: {img_path.name} | Área px: {area_en_pixeles} | Área real: {area_en_cm2:.4f} cm²")

    count += 1

print(f"--- Proceso completado ---")
print(f"Se procesaron {count} imágenes.")